In [0]:
import requests

URL = "https://data.gharchive.org/2026-09-20-12.json.gz"
try:
    r = requests.head(URL, timeout=15, allow_redirects=True)
    print("status:", r.status_code)
    print("size (MB):", round(int(r.headers.get("Content-Length", 0)) / 1e6, 1))
except Exception as e:
    print("BLOCKED / ERROR:", type(e).__name__, e)

In [0]:
import os, time, logging, requests

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger("opspulse.download")

LANDING_PATH = "/Volumes/opspulse/bronze/landing"
HOURS = [("2026-09-20", h) for h in (12, 13, 14)]

def download_hour(date: str, hour: int, retries: int = 3) -> str:
    """Ek ghante ki file Volume mein rakhta hai. Pehle se maujood ho to skip (idempotent)."""
    url = f"https://data.gharchive.org/{date}-{hour}.json.gz"   # hour bina leading zero
    folder = f"{LANDING_PATH}/date={date}/hour={hour:02d}"
    final = f"{folder}/{date}-{hour}.json.gz"
    tmp = final + ".part"
    os.makedirs(folder, exist_ok=True)

    if os.path.exists(final) and os.path.getsize(final) > 0:
        log.info("skip (exists): %s", final)
        return final

    for attempt in range(1, retries + 1):
        try:
            with requests.get(url, stream=True, timeout=60) as r:
                r.raise_for_status()
                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        f.write(chunk)
            os.replace(tmp, final)   # poori file aane ke baad hi final naam
            log.info("downloaded: %s (%.1f MB)", final, os.path.getsize(final) / 1e6)
            return final
        except Exception as e:
            log.warning("attempt %d/%d failed for %s: %s", attempt, retries, url, e)
            time.sleep(2 * attempt)
    raise RuntimeError(f"download failed after {retries} attempts: {url}")

for d, h in HOURS:
    download_hour(d, h)

In [0]:
for root, _, files in os.walk(LANDING_PATH):
    for f in files:
        p = os.path.join(root, f)
        print(f"{os.path.getsize(p)/1e6:8.1f} MB  {p}")